In [3]:

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

In [5]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print(f"Dataset shape: {X.shape}")
print(f"Target classses: {list(data.target_names)}\n")

Dataset shape: (569, 30)
Target classses: [np.str_('malignant'), np.str_('benign')]



In [6]:
X_test, X_train, y_test, y_train = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000, random_state=42))
])

rf_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

gb_pipeline = Pipeline([
    ('clf', GradientBoostingClassifier(random_state=42))
])

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('--- Cross-Validation Scores ---\n')
for name, model in [('Logistic Regression', lr_pipeline), 
                    ('Random Forest', rf_pipeline), 
                    ('Gradient Boosting', gb_pipeline)]:
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    print(f"{name}: Mean Accuracy: {scores.mean():.4f}, Std: {scores.std():.4f}\n")

--- Cross-Validation Scores ---

Logistic Regression: Mean Accuracy: 0.9561, Std: 0.0275

Random Forest: Mean Accuracy: 0.9387, Std: 0.0442

Gradient Boosting: Mean Accuracy: 0.9213, Std: 0.0637



In [10]:
param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [None, 5, 10],
    'clf__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\n--- Hyperparameter Tuning ---")
print("Best Parameters found:", grid_search.best_params_)
print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}")


--- Hyperparameter Tuning ---
Best Parameters found: {'clf__max_depth': None, 'clf__min_samples_split': 2, 'clf__n_estimators': 100}
Best Cross-Validation Score: 0.9387


In [11]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n--- Classification Report on Test Set ---")
print(classification_report(y_test, y_pred, target_names=data.target_names))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))


--- Classification Report on Test Set ---
              precision    recall  f1-score   support

   malignant       0.97      0.90      0.94       170
      benign       0.94      0.99      0.96       285

    accuracy                           0.95       455
   macro avg       0.96      0.94      0.95       455
weighted avg       0.95      0.95      0.95       455

--- Confusion Matrix ---
[[153  17]
 [  4 281]]
